<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_9_Simple_Parallelism_for_Faster_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import logging
import pandas as pd
import time

In [ ]:
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [ ]:
import time
import logging

def process_item(x: int) -> int:
    # simulating slow work
    time.sleep(1)
    return x * 2

if __name__ == "__main__":
    logger = logging.getLogger("__main__")
    logging.basicConfig(level=logging.INFO)

    start = time.perf_counter()
    results = [process_item(x) for x in range(5)]
    end = time.perf_counter()

    logger.info(results)
    logger.info(f"Time taken: {end - start:.2f} seconds")

INFO:__main__:[0, 2, 4, 6, 8]
INFO:__main__:Time taken: 5.00 seconds


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def process_item(x: int) -> int:
    # simulate slow work
    time.sleep(1)
    return x * 2

if __name__ == "__main__":
    start = time.perf_counter()

    with ThreadPoolExecutor() as executor:
        results = list(executor.map(process_item, range(5)))

    end = time.perf_counter()

    logger.info(results)
    logger.info(f"Time taken: {end - start:.2f} seconds")

INFO:__main__:[0, 2, 4, 6, 8]
INFO:__main__:Time taken: 1.00 seconds


## **ThreadPoolExecutor With default settings**

In [ ]:
from typing import List, Dict, Optional
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor
import logging
import time

def fetch_user_from_api(user_id: int) -> Optional[Dict]:
    """Fetch a user from JSONPlaceholder API."""
    try:
        url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        logger.error(f"Timeout fetching user {user_id}")
        return None

def fetch_users_in_parallel(user_ids: List[int]) -> pd.DataFrame:
    """Fetch multiple users from an external API in parallel."""

    with ThreadPoolExecutor() as executor:
        results = list(executor.map(fetch_user_from_api, user_ids))
    # Filter out failed requests
    valid_results = [r for r in results if r is not None]

    if len(valid_results) < len(user_ids):
        logger.warning(f"Successfully fetched {len(valid_results)}/{len(user_ids)} users")
    return pd.DataFrame(valid_results)

def fetch_data_in_parallel_helper(user_ids: list[int]) -> None:
    """Helper to run parallel fetch and show timing."""
    logger.info("Starting parallel fetch...")
    start_time = time.perf_counter()

    users_df = fetch_users_in_parallel(user_ids)

    duration = time.perf_counter() - start_time
    logger.info(f"Successfully fetched {len(users_df)} users in {duration:.2f} seconds")
    return users_df

# Fetch data for multiple users in parallel
if __name__ == "__main__":
    logger = logging.getLogger("__main__")
    logging.basicConfig(level=logging.INFO)
    user_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    df=fetch_data_in_parallel_helper(user_ids)
    logger.info(f"DataFrame shape: {df.shape}")

INFO:__main__:Starting parallel fetch...
INFO:__main__:Successfully fetched 10 users in 0.07 seconds
INFO:__main__:DataFrame shape: (10, 8)


## **With Max_workers and as_completed**

In [ ]:
from typing import List, Dict, Optional
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import time

logger = logging.getLogger(__name__)

def fetch_user_from_api(user_id: int) -> Optional[Dict]:
    """Fetch a user from JSONPlaceholder API."""
    try:
        url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        logger.error(f"Timeout fetching user {user_id}")
        return None

def fetch_users_with_control(
    user_ids: List[int],
    max_workers: int = 8
) -> pd.DataFrame:
    """Fetch users with controlled parallelism and flexible result handling."""

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_id = {
            executor.submit(fetch_user_from_api, uid): uid
            for uid in user_ids if uid is not None
        }
        for future in as_completed(future_to_id):
            user_id = future_to_id[future]
            data = future.result()
            if data:
                results.append(data)

    if len(results) < len(user_ids):
        logger.warning(f"Successfully fetched {len(results)}/{len(user_ids)} users")

    return pd.DataFrame(results)

def fetch_data_in_parallel_helper(user_ids: list[int]) -> pd.DataFrame:
    """Helper to run parallel fetch and show timing."""
    logger.info("Starting parallel fetch...")
    start_time = time.time()

    users_df = fetch_users_with_control(user_ids)

    duration = time.time() - start_time
    logger.info(f"Successfully fetched {len(users_df)} users in {duration:.2f} seconds")
    return users_df

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    user_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    df = fetch_data_in_parallel_helper(user_ids)
    logger.info(f"DataFrame shape: {df.shape}")

INFO:__main__:Starting parallel fetch...
INFO:__main__:Successfully fetched 10 users in 0.09 seconds
INFO:__main__:DataFrame shape: (10, 8)


## **Sequential version**

In [ ]:
from typing import List, Dict, Optional
import pandas as pd
import requests
import logging
import time

logger = logging.getLogger(__name__)

def fetch_user_from_api(user_id: int) -> Optional[Dict]:
    """Fetch a user from JSONPlaceholder API."""
    try:
        url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        logger.error(f"Timeout fetching user {user_id}")
        return None

def fetch_users_sequential(user_ids: List[int]) -> pd.DataFrame:
    """Sequential version — much slower for many requests."""
    results = []
    for uid in user_ids:
        data = fetch_user_from_api(uid)
        if data:
            results.append(data)
    return pd.DataFrame(results)

# Fetching data in a sequence
def fetch_data_in_sequence_helper(user_ids: list):
    logger.info("Starting sequential fetch...")
    start_time = time.time()
    users_df = fetch_users_sequential(user_ids)
    duration = time.time() - start_time
    logger.info(f"Successfully fetched {len(users_df)} users in {duration:.2f} seconds")
    return users_df

if __name__ == "__main__":
    # Fetch data for multiple users in sequence
    logging.basicConfig(level=logging.INFO)
    user_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    df = fetch_data_in_sequence_helper(user_ids)
    logger.info(f"DataFrame shape: {df.shape}")

INFO:__main__:Starting sequential fetch...
INFO:__main__:Successfully fetched 10 users in 0.34 seconds
INFO:__main__:DataFrame shape: (10, 8)


## **Streaming + Parallelism**

In [ ]:
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import islice
from typing import Dict, Iterator, Optional
import requests

logger = logging.getLogger(__name__)

def fetch_user_from_api(user_id: int) -> Optional[Dict]:
    """Fetch a user from JSONPlaceholder API."""
    try:
        url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        logger.error(f"Timeout fetching user {user_id}")
        return None

def generate_user_ids(n: int) -> Iterator[int]:
    """Yield user IDs lazily."""
    for i in range(1, n + 1):
        yield i

def fetch_users_resilient_stream(
    user_ids: Iterator[int],
    chunk_size: int = 20,
    max_workers: int = 8
) -> Iterator[Dict]:
    """Parallel streaming with per-item error."""

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        while chunk := list(islice(user_ids, chunk_size)):
            future_to_uid = {
                executor.submit(fetch_user_from_api, uid): uid
                for uid in chunk
            }

            for future in as_completed(future_to_uid):
                result = future.result()
                if result:
                    yield result

# Pipeline execution
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    user_stream = generate_user_ids(10)

    for user in fetch_users_resilient_stream(user_stream):
        logger.info(f"Processed User: {user.get('name')}")

INFO:__main__:Processed User: Leanne Graham
INFO:__main__:Processed User: Chelsey Dietrich
INFO:__main__:Processed User: Patricia Lebsack
INFO:__main__:Processed User: Kurtis Weissnat
INFO:__main__:Processed User: Mrs. Dennis Schulist
INFO:__main__:Processed User: Ervin Howell
INFO:__main__:Processed User: Clementine Bauch
INFO:__main__:Processed User: Nicholas Runolfsdottir V
INFO:__main__:Processed User: Glenna Reichert
INFO:__main__:Processed User: Clementina DuBuque


## **ProcessPoolExecutor**

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from typing import List
import time

def cpu_heavy_task(n: int) -> int:
    """Simulate a CPU-intensive calculation."""
    result = 0
    for i in range(n * 1_000_000):
        result += i
    return result

def run_cpu_parallel(numbers: List[int], max_workers: int = 4) -> List[int]:
    """Run CPU-bound tasks in parallel using separate processes."""
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(cpu_heavy_task, numbers))
    return results

def run_cpu_sequential(numbers: List[int]) -> List[int]:
    """Sequential version for comparison."""
    return [cpu_heavy_task(n) for n in numbers]


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    numbers = [10, 20, 30, 40, 50]

    # Sequential execution
    start = time.perf_counter()
    seq_results = run_cpu_sequential(numbers)
    seq_time = time.perf_counter() - start

    # Parallel execution
    start = time.time()
    par_results = run_cpu_parallel(numbers, max_workers=4)
    par_time = time.time() - start

    logger.info(f"Sequential time: {seq_time:.2f} seconds")
    logger.info(f"Parallel time (4 processes): {par_time:.2f} seconds")
    logger.info(f"Speedup: {seq_time / par_time:.2f}x")
    logger.info(f"Results match: {seq_results == par_results}")

INFO:__main__:Sequential time: 9.85 seconds
INFO:__main__:Parallel time (4 processes): 10.39 seconds
INFO:__main__:Speedup: 0.95x
INFO:__main__:Results match: True
